In [245]:
#!pip install pytorch-forecasting --extra-index-url https://download.pytorch.org/whl/cpu

In [246]:
import numpy as np
import pandas as pd
from pytorch_forecasting import TimeSeriesDataSet

In [247]:
class ZillowHousingDataService:
    
    def __init__(self, new_construction_price_per_square_foot_data_set_empty):
        self.new_construction_price_per_square_foot_data_set_formatted = new_construction_price_per_square_foot_data_set_empty
        
    def get_new_construction_price_per_square_foot_data_set_formatted(self):
        return self.new_construction_price_per_square_foot_data_set_formatted
        
    def get_new_construction_price_per_square_foot_data_set_training(self, region_ids=None):
        return None
        
    def format_new_construction_price_per_square_foot_data_set(self, create_repeating_array_function, data_frames, region_ids=None):
        
        if region_ids == None:
            index_values = data_frames.index
            
        else:
            index_values = region_ids
            
        for index_value in index_values:
            
            new_data_frame = data_frames.loc[[index_value]]
            
            time_series_data = new_data_frame.get(data_frames.columns[4:len(data_frames.columns)])
            
            time_series_data = time_series_data.set_axis(["Price"], axis=0)
            
            new_data_frame.drop(columns=new_data_frame.columns, inplace=True)
            
            new_data_frame = new_data_frame.assign(RegionID=[create_repeating_array_function(index_value, len(time_series_data.columns))])
            
            new_data_frame = new_data_frame.explode("RegionID")
            
            new_data_frame.reset_index(drop=True, inplace=True)
            
            new_data_frame = new_data_frame.assign(Date=time_series_data.columns.T)
            
            new_data_frame = new_data_frame.assign(Price=time_series_data.T.to_numpy(copy=True))
            
            self.new_construction_price_per_square_foot_data_set_formatted[index_value] = new_data_frame

In [248]:
data_file_path = "~/ece492-final-project/"
data_file_name = "Metro_new_con_median_sale_price_per_sqft_uc_sfrcondo_month.csv"
data_frames = pd.read_csv(data_file_path + data_file_name)
data_frames.set_index("RegionID", inplace=True)
data_frames.sort_index(inplace=True)

In [249]:
data_service = ZillowHousingDataService({})
data_service.format_new_construction_price_per_square_foot_data_set(np.repeat, data_frames) #data_frames.index[0:1:1]
formatted_data_sets = data_service.get_new_construction_price_per_square_foot_data_set_formatted()
#print(formatted_data_sets)
print(formatted_data_sets[102001])

   RegionID        Date       Price
0    102001  2018-01-31  134.134666
1    102001  2018-02-28  134.826025
2    102001  2018-03-31  136.998984
3    102001  2018-04-30  136.775042
4    102001  2018-05-31  139.170897
..      ...         ...         ...
92   102001  2025-09-30  204.545455
93   102001  2025-10-31  206.698440
94   102001  2025-11-30  203.475870
95   102001  2025-12-31  201.694232
96   102001  2026-01-31  201.995012

[97 rows x 3 columns]


In [250]:
# get data sets with take first half of indices
